# 01 - Conexão com o Data Lake (ADLS Gen2)
**Squad 1 — Data Quality em Tempo Real | Dupla 1**
**Integrantes:** Gabriel Franz Simoni & Carlos Eduardo Santos de Souza
**Tabelas:** `ecommerce_itens_pedido` e `ecommerce_rastreamento_entregas`

### Objetivo (Task 2):
Carregar credenciais do Secret Scope, preparar as opções OAuth para leitura nativa via Spark (`spark.read.options`), e validar a conexão listando o conteúdo do container.

**Pré-requisito:** `00_setup_secrets.ipynb` já executado (secrets já existem no scope `internship-squad1`).

In [0]:
adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

adls_options = {
    f"fs.azure.account.auth.type.{adls_storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_storage_account}.dfs.core.windows.net": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{adls_storage_account}.dfs.core.windows.net": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{adls_storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

print(f"Opções OAuth preparadas para {adls_storage_account}")

In [0]:
caminho_validacao = f"abfss://raw@{adls_storage_account}.dfs.core.windows.net/real-time-data/*/*/*/*/ecommerce_itens_pedido.parquet"

df_validacao = spark.read.options(**adls_options).parquet(caminho_validacao)
print(f"Conexão validada: {df_validacao.count()} linhas encontradas em ecommerce_itens_pedido (todas as janelas).")